# TP 7: Introducción a Databricks y procesamiento de datos con Apache Spark
**Materia:** Análisis de Datos en la Nube
**Integrantes:** Páez, Simon, Villanueva, Vilches
**Dataset:** Iris (150 observaciones, 4 variables numéricas y la especie)
**Entorno:** Databricks Free Edition, Apache Spark y Delta Lake

## Etapa 1: Configuración del entorno

Para empezar con este trabajo practico, primero verificamos si el entorno funciona de manera correcta

In [0]:
# Comando simple en Python para verificar que el notebook está conectado al cómputo
print ("El entorno esta listo")

## Etapa 2: Carga y exploración de datos

In [0]:
#Primero leemos el CSV previamente cargado desde Volumes usando Spark

ruta_archivo = "/Volumes/workspace/default/datos_iris/Iris.csv"

df_raw = (spark.read
          .option("header", "true")      
          .option("inferSchema", "true")  # Detecta automáticamente números y texto
          .csv(ruta_archivo))


# Mostramos el esquema y las primeras 10 filas
df_raw.printSchema()
display(df_raw.limit(10))

In [0]:
# Registramos el DataFrame como una vista temporal para usar Spark SQL
df_raw.createOrReplaceTempView("iris_vista")
print ("Vista temporal creada correctamente")


In [0]:
%sql
-- 1. Ver las primeras 10 filas usando SQL
SELECT * FROM iris_vista LIMIT 10 ;

In [0]:
%sql
-- 2. Contar registros y calcular promedios por especie
SELECT 
    Species, 
    COUNT(*) AS total_flores,
    ROUND(AVG(SepalLengthCm),2) AS promedio_largo_sepalo,
    ROUND(AVG(PetalLengthCm),2) AS promedio_largo_petalo
FROM iris_vista
GROUP BY Species
ORDER BY total_flores DESC

## Etapa 3: Transformaciones de datos
Se eliminan duplicados y nulos, se crean columnas calculadas, se filtra, se agrega por especie y se hace un join.

In [0]:
# Limpieza de datos (duplicados y nulos)

from pyspark.sql import functions as F

#Duplicados: Ignoramos ID porque es unico por fila y ocultaria los repetidos
df_limpio = df_raw.dropDuplicates([c for c in df_raw.columns if c != 'Id'])

print ("Filas sin duplicados: ", df_limpio.count())

# Nulos: contar por columna y luego eliminar/rellenar

print ("Nulos por columna:")
df_limpio.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_limpio.columns]).show()

df_limpio = df_limpio.dropna()

In [0]:
# Limpiar el nombre de la especie ("Iris-setosa" -> "setosa") y renombrar columnas
df_t = (df_limpio
    .withColumn("Species", F.regexp_replace("Species", "Iris-", ""))
    .withColumnRenamed("SepalLengthCm", "sepal_length")
    .withColumnRenamed("SepalWidthCm", "sepal_width")
    .withColumnRenamed("PetalLengthCm", "petal_length")
    .withColumnRenamed("PetalWidthCm", "petal_width")
    # Columnas calculadas
    .withColumn("petal_area", F.round(F.col("petal_length") * F.col("petal_width"), 2))
    .withColumn("sepal_area", F.round(F.col("sepal_length") * F.col("sepal_width"), 2))
    .withColumn("petal_ratio", F.round(F.col("petal_length") / F.col("petal_width"), 2))
    # Categoría con condicional
    .withColumn("tamano_petalo",
        F.when(F.col("petal_length") < 2, "chico")
         .when(F.col("petal_length") < 5, "mediano")
         .otherwise("grande"))
)

# Filtro: flores con sépalo largo
df_t.filter(F.col("sepal_length") > 6.0).show(5)
df_t.show(5)

In [0]:
df_agg = (df_t.groupBy("Species")
    .agg(F.count("*").alias("cantidad"),
         F.round(F.avg("sepal_length"), 2).alias("prom_sepal_length"),
         F.round(F.avg("petal_length"), 2).alias("prom_petal_length"),
         F.round(F.avg("petal_width"), 2).alias("prom_petal_width"),
         F.round(F.avg("petal_area"), 2).alias("prom_petal_area"))
    .orderBy("Species"))
df_agg.show()

In [0]:
# Creamos una tabla de referencia con informacion de cada especie y la unimos

info = spark.createDataFrame([
    ("setosa",     "Iris setosa",     "Alaska, Siberia"),
    ("versicolor", "Iris versicolor", "Norteamérica"),
    ("virginica",  "Iris virginica",  "Este de EE. UU."),
], ["Species", "nombre_cientifico", "region"])

df_join = df_t.join(info, on="Species", how="left")
df_join.select("Id", "Species", "nombre_cientifico", "region", "petal_length").show(5)

## Etapa 4: Almacenamiento con Delta Lake


In [0]:
# Guardamos los datos transformados en formato Delta (tabla administrada)
df_join.write.format("delta").mode("overwrite").saveAsTable("iris_delta")
print("Tabla 'iris_delta' guardada en formato Delta")

### Validación de la integridad
Se lee la tabla Delta y se compara con el DataFrame original.


In [0]:
# Leemos nuevamente desde Delta Lake
df_delta = spark.table("iris_delta")

# 1) Misma cantidad de filas
print("Origen:", df_join.count(), "| Delta:", df_delta.count())

# 2) Mismo esquema (comparamos nombres y tipos; Delta puede cambiar el atributo nullable)
print("Esquemas iguales:",
      [(f.name, f.dataType) for f in df_join.schema] == [(f.name, f.dataType) for f in df_delta.schema])

# 3) Sin diferencias de contenido
print("Filas distintas:", df_join.exceptAll(df_delta).count())

# 4) Característica propia de Delta: historial de versiones
spark.sql("DESCRIBE HISTORY iris_delta").select("version", "timestamp", "operation").show()

### Actualización y time travel
Delta guarda cada cambio como una versión nueva. Actualizamos las flores con pétalo menor a 1.5 cm (categoría "muy chico") y comparamos la versión original (0) con la actual.

In [0]:
%sql
-- Actualizamos algunas filas para generar una nueva versión

UPDATE iris_delta SET tamano_petalo = 'muy chico' WHERE petal_length < 1.5;

In [0]:
%sql
-- Actualizamos algunas filas para generar una nueva versión de la tabla
UPDATE iris_delta SET tamano_petalo = 'muy chico' WHERE petal_length < 1.5;

In [0]:
%sql
-- Time travel: conteo por categoría en la versión original (0)
SELECT tamano_petalo, COUNT(*) AS cantidad FROM iris_delta VERSION AS OF 0 GROUP BY tamano_petalo ORDER BY tamano_petalo;

In [0]:
%sql
-- Conteo por categoría en la versión actual
SELECT tamano_petalo, COUNT(*) AS cantidad FROM iris_delta GROUP BY tamano_petalo ORDER BY tamano_petalo;

In [0]:
%sql
-- Historial completo de versiones de la tabla
DESCRIBE HISTORY iris_delta;

**Interpretación:** en la versión 0 no existe la categoría "muy chico"; en la versión actual sí, y esas filas salieron de la categoría "chico". El historial muestra la versión 0 (creación de la tabla) y la versión 1 (actualización), y Delta permite consultar cualquiera de las dos.

## Etapa 5: Visualización y análisis
Los gráficos se hacen con matplotlib a partir de la tabla Delta.

Los comparamos con los mismos graficos creados mediante la interfaz de Databricks

In [0]:
import matplotlib.pyplot as plt

# Releemos la tabla Delta (versión actual) y la convertimos a pandas
pdf = spark.table("iris_delta").toPandas().sort_values("Id")

# Un color fijo por especie para que todos los gráficos sean consistentes
colores = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}
print(pdf.shape)
pdf.head()

In [0]:
# Promedio de cada medida agrupado por especie (se calcula con pandas sobre el DataFrame)
prom = pdf.groupby("Species")[["sepal_length", "petal_length", "petal_width"]].mean().round(2)
prom.columns = ["Largo del sépalo", "Largo del pétalo", "Ancho del pétalo"]

ax = prom.plot(kind="bar", figsize=(9, 5), edgecolor="k")
ax.set_title("Promedio de medidas por especie")
ax.set_xlabel("Especie")
ax.set_ylabel("Medida promedio (cm)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)

# Mostramos el valor sobre cada barra
for contenedor in ax.containers:
    ax.bar_label(contenedor, fontsize=8, padding=2)

plt.tight_layout()
plt.show()

In [0]:
display(df_agg)

Databricks visualization. Run in Databricks to view.

**Interpretación:** las tres medidas aumentan de setosa a versicolor y a virginica. La diferencia es mucho más fuerte en el pétalo (largo de 1.46 a 5.56 cm) que en el sépalo (de 5.01 a 6.6 cm).

### Gráfico 2: Dispersión, largo vs ancho del pétalo

In [0]:
# Cada punto es una flor; el color indica la especie
fig, ax = plt.subplots(figsize=(8, 5))

for especie, grupo in pdf.groupby("Species"):
    ax.scatter(grupo["petal_length"], grupo["petal_width"],
               label=especie, color=colores[especie],
               alpha=0.7, edgecolor="k", s=50)

ax.set_title("Iris: largo vs ancho del pétalo por especie")
ax.set_xlabel("Largo del pétalo (cm)")
ax.set_ylabel("Ancho del pétalo (cm)")
ax.legend(title="Especie")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
display(df_delta)

Databricks visualization. Run in Databricks to view.

**Interpretación:** setosa forma un grupo aislado (pétalos chicos), totalmente separado de las otras dos. Versicolor y virginica están más arriba y a la derecha, con una zona de superposición pequeña. Con solo el tamaño del pétalo ya se puede separar bien las especies.

### Gráfico 3: Líneas, medidas por observación (Id)


In [0]:
# Las filas del dataset están ordenadas por especie, por eso el gráfico muestra "escalones"
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(pdf["Id"], pdf["petal_length"], label="Largo del pétalo")
ax.plot(pdf["Id"], pdf["petal_width"],  label="Ancho del pétalo")
ax.plot(pdf["Id"], pdf["sepal_length"], label="Largo del sépalo")

# Líneas verticales donde cambia la especie (Id 1-50 setosa, 51-100 versicolor, 101-150 virginica)
for x in [50, 100]:
    ax.axvline(x, color="gray", linestyle="--", alpha=0.6)

# Etiquetas de cada especie en la parte superior del gráfico
ymax = pdf[["petal_length", "petal_width", "sepal_length"]].max().max()
for xpos, nombre in [(25, "setosa"), (75, "versicolor"), (125, "virginica")]:
    ax.text(xpos, ymax + 0.2, nombre, ha="center", fontsize=10)

ax.set_title("Medidas de las flores por observación")
ax.set_xlabel("Id de la observación")
ax.set_ylabel("Medida (cm)")
ax.set_ylim(0, ymax + 0.8)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
display(df_delta.orderBy("Id"))

Databricks visualization. Run in Databricks to view.

**Interpretación:** el largo del pétalo es bajo y casi plano en setosa, sube en versicolor y vuelve a subir en virginica. El largo del sépalo cambia mucho menos. Faltan 3 Id porque se eliminaron duplicados; no afecta la lectura.

## Etapa 6: Entrega final


**Proceso:** se cargó el dataset Iris (150 observaciones originales) en Databricks desde un Volume y se exploró con Spark SQL mediante una vista temporal. Luego se limpió: se eliminaron 3 filas duplicadas (quedaron 147) y no había nulos. Se crearon columnas calculadas (áreas del pétalo y sépalo, relación largo/ancho y categoría de tamaño), se filtró, se agregó por especie y se hizo un join con una tabla de referencia. El resultado se guardó en formato Delta Lake y se validó releyéndolo (misma cantidad de filas, mismo esquema en nombres y tipos, y sin diferencias de contenido). Se probó además una actualización con time travel entre versiones. Por último se hicieron tres gráficos con matplotlib: barras, dispersión y líneas.

**Resultados y conclusiones:**
- Setosa se distingue claramente: su largo promedio de pétalo es 1.46 cm, contra 4.26 cm en versicolor y 5.56 cm en virginica.
- Versicolor y virginica son parecidas entre sí, aunque virginica tiene pétalos más grandes (ancho promedio de 2.03 cm contra 1.33 cm, y área promedio de 11.33 contra 5.72).
- Las medidas del pétalo separan mejor las especies que las del sépalo, cuyo largo promedio varía solo de 5.01 a 6.6 cm.
- El gráfico de dispersión muestra a setosa como un grupo aislado y a las otras dos con una pequeña superposición.
- Delta Lake permitió guardar los datos de forma confiable, validar la integridad y consultar versiones anteriores de la tabla.